## CHARGEMENT LOCAL VERS LE BRONZE

In [ ]:
from pyspark.sql import SparkSession
from pyspark import SparkConf
import pandas as pd
import os

#LORSQU'ON TRAVAILLE DEPUIS SA MACHINE LOCAL
MINIO_ENDPOINT = "http://192.168.1.230:30137"
MINIO_ACCESS_KEY = "datalab-team"
MINIO_SECRET_KEY = "minio-datalabteam123"
NESSIE_URI = "http://192.168.1.230:30604/api/v1" 

#---------------------------------------------------------------------------------

#LORSQU'ON TRAVAILLE SUR JHUB
MINIO_ENDPOINT = "http://minio.mon-namespace.svc.cluster.local:80"
MINIO_ACCESS_KEY = "datalab-team"
MINIO_SECRET_KEY = "minio-datalabteam123"
NESSIE_URI = "http://nessie.trino.svc.cluster.local:19120/api/v1" 

#---------------------------------------------------------------------------------

conf = (
    SparkConf()
    .setAppName("Job_Scrapping")
    .set("spark.driver.host", "127.0.0.1")
    .set("spark.driver.bindAddress", "127.0.0.1")
    .set("spark.driver.memory", "16g")
    # AJOUT DES PACKAGES : On ajoute le jar nessie-spark-extensions
    .set("spark.jars.packages", 
         "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1,"
         "org.apache.hadoop:hadoop-aws:3.3.4,"
         "org.projectnessie.nessie-integrations:nessie-spark-extensions-3.5_2.12:0.77.1")
     
     .set("spark.sql.extensions", 
         "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
         "org.projectnessie.spark.extensions.NessieSparkSessionExtensions")

    # CONFIGURATION DU CATALOGUE NESSIE
    .set("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog")
    .set("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .set("spark.sql.catalog.nessie.uri", NESSIE_URI)
    .set("spark.sql.catalog.nessie.ref", "main")
    
    # On définit le bucket bronze comme entrepôt par défaut du catalogue
    .set("spark.sql.catalog.nessie.warehouse", "s3a://bronze/")
    
    # CONFIGURATION MINIO (S3A)
    .set("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .set("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .set("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
    # set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
)


In [3]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()

:: loading settings :: url = jar:file:/Users/datalab/spark_env/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/datalab/.ivy2/cache
The jars for the packages stored in: /Users/datalab/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-86fc23cd-f498-4f6f-af99-4ab634628238;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_2.12;0.77.1 in central
:: resolution report :: resolve 74ms :: artifacts dl 4ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from centr

In [ ]:
pdf = pd.read_excel("<<CHEMIN VERS FICHIER EN LOCAL>>")
df_spark = spark.createDataFrame(pdf)

for col in df_spark.columns:
    df_spark = df_spark.withColumnRenamed(col, col.replace(" ", "_").replace(".", "_"))

## ON EFFECTUE LES TRAITEMENTS PERSONNALISES ICI

In [ ]:
table_target = "nessie.bronze.<<nom de la table>>"

spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.bronze")

df_spark.write \
    .format("iceberg") \
    .mode("overwrite") \
    .saveAsTable(table_target)

26/01/22 02:02:04 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1081186 ms exceeds timeout 120000 ms
26/01/22 02:02:04 WARN SparkContext: Killing executors is not supported by current scheduler.
26/01/22 02:02:08 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$